# Retrieval-Augmented Generation (RAG)

In [1]:
from setup import bedrock_tool
import chromadb
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

✅ AWS credentials found
✅ OpenAI credentials found
✅ EXA credentials found


Create a static calorie table that we can use as a tool:

In [8]:
# We populated the RAG with the data from the data/calories.csv file in
# the ../rag_setup/rag_setup.ipynb notebook

chroma_client = chromadb.PersistentClient("../chroma")
nutrition_qna = chroma_client.get_collection(name="nutrition_qna")

In [11]:
results = nutrition_qna.query(query_texts=["Is rice good for weight loss?"], n_results=2)
for i, doc in enumerate(results["documents"][0]):   
    print(doc)
    print("\n")

Question: Which ingredients in lemon rice make it beneficial for one's diet?
        Answer: The inclusion of peanuts, which are high in protein, makes lemon rice a nutritious choice.

        This Q&A pair provides information about nutrition and health topics.


Question: What's a misconception about the calorie content of starchy food items like rice?
        Answer: Many people think these foods are naturally high in calories, but that's not true. Plain versions of them are not very high in calories.

        This Q&A pair provides information about nutrition and health topics.




In [16]:
@function_tool
def nutrition_qna_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for answering health-related nutritional questions
    using the 'nutrition_qna' ChromaDB collection.

    Args:
        query: The nutrition or health-related question.
        max_results: The maximum number of relevant answers to return.

    Returns:
        A formatted string containing relevant nutritional information.
    """

    results = nutrition_qna.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No relevant information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        source = metadata.get("source", "Nutrition Database")
        category = metadata.get("category", "General")

        formatted_results.append(
            f"{source} ({category}): {source}"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `calorie_lookup_tool` function_

In [17]:
# print(nutrition_qna_tool('rice'))

In [18]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out reply for the question about nutrition.
    You give concise answers.
    If you need to look up  information, use the nutrition_qna_tool.
    """,
    model="litellm/bedrock/eu.amazon.nova-lite-v1:0",
    tools=[bedrock_tool(nutrition_qna_tool.__dict__)],
)

In [19]:
with trace("Nutrition Assistant with RAG") as t:
    result = await Runner.run(
        calorie_agent,
        "Is rice good for weight loss?",
    )

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: 

Rice can be part of a weight loss diet, but it depends on the type and portion size. Brown rice is generally considered healthier as it is a whole grain and contains more fiber, which can help you feel full. White rice is more processed and has less fiber, which may lead to overeating. Both types of rice have similar calorie content per cup when cooked, but portion control is key. 

To aid weight loss, pair rice with plenty of vegetables and lean proteins, and be mindful of your total calorie intake.

Trace URL: https://platform.openai.com/logs/trace?trace_id=trace_2c959afe6d234384a576d49842969524
